# Instructor Effectiveness Modeling

This project aims to predict instructor effectiveness tiers from batch-level EdTech data. Each row in the dataset represents an instructor–batch–course combination and includes metrics such as completion rate, score improvement, quiz scores, dropout rate, watch time, assignment submission rate, forum activity, and feedback scores. By engineering meaningful features and training classification models, we will categorize instructors into effectiveness tiers (e.g., Low, Medium, High) and identify the key drivers that distinguish highly effective instructors from the rest.

In [1]:
# ── Imports ──────────────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Reproducibility
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

print('All imports loaded successfully.')

All imports loaded successfully.


In [2]:
# ── Load the dataset ─────────────────────────────────────────────────────────
df = pd.read_csv('data.csv')
print(f'Dataset loaded: {df.shape[0]} rows × {df.shape[1]} columns')

Dataset loaded: 2000 rows × 12 columns


In [3]:
# ── Data inspection ──────────────────────────────────────────────────────────

# Shape
print('='*60)
print('SHAPE')
print('='*60)
print(df.shape)
print()

# First few rows
print('='*60)
print('HEAD')
print('='*60)
df.head()

SHAPE
(2000, 12)

HEAD


,batch_id,instructor_id,course_id,completion_rate,avg_score_improvement,avg_quiz_score,dropout_rate,avg_watch_time,assignment_submission_rate,forum_activity_rate,avg_feedback_score,feedback_response_rate
0,B_1861,I_044,C_01,0.300000,14.225955,73.546528,0.647423,0.774572,0.790918,0.108414,3.766211,0.533193
1,B_0354,I_119,C_06,0.657220,22.871110,77.312331,0.425098,0.494936,0.998566,0.280550,5.000000,0.734087
2,B_1334,I_050,C_03,0.300000,16.087517,79.563687,0.700000,0.977901,0.807298,0.207013,3.517386,0.681433
3,B_0906,I_024,C_21,0.639507,24.260687,99.295316,0.334657,0.846515,0.544555,0.306395,4.207578,1.000000
4,B_1290,I_001,C_08,0.527302,31.081556,99.393425,0.521099,0.917450,0.865885,0.252289,4.426230,0.696710


In [4]:
# Info
print('='*60)
print('INFO')
print('='*60)
df.info()

INFO
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 12 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   batch_id                    2000 non-null   object 
 1   instructor_id               2000 non-null   object 
 2   course_id                   2000 non-null   object 
 3   completion_rate             2000 non-null   float64
 4   avg_score_improvement       2000 non-null   float64
 5   avg_quiz_score              2000 non-null   float64
 6   dropout_rate                2000 non-null   float64
 7   avg_watch_time              2000 non-null   float64
 8   assignment_submission_rate  2000 non-null   float64
 9   forum_activity_rate         2000 non-null   float64
 10  avg_feedback_score          2000 non-null   float64
 11  feedback_response_rate      2000 non-null   float64
dtypes: float64(9), object(3)
memory usage: 187.6+ KB


In [5]:
# Descriptive statistics
print('='*60)
print('DESCRIBE')
print('='*60)
df.describe()

DESCRIBE


,completion_rate,avg_score_improvement,avg_quiz_score,dropout_rate,avg_watch_time,assignment_submission_rate,forum_activity_rate,avg_feedback_score,feedback_response_rate
count,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000,2000.000000
mean,0.602808,27.035844,77.956126,0.394883,0.776515,0.753188,0.250300,4.207134,0.736519
std,0.159667,5.716641,10.695618,0.162747,0.145231,0.148058,0.100640,0.419209,0.149412
min,0.300000,6.159240,40.386725,0.020000,0.287440,0.251111,0.000000,2.639915,0.259935
25%,0.489260,23.124673,70.897590,0.280035,0.675076,0.652110,0.179845,3.918986,0.633293
50%,0.603091,26.938629,78.020567,0.394820,0.780330,0.756380,0.249771,4.205989,0.737213
75%,0.712797,30.885600,85.444286,0.511432,0.894242,0.856458,0.319204,4.503437,0.845876
max,0.980000,40.000000,100.000000,0.700000,1.000000,1.000000,0.641111,5.000000,1.000000


In [6]:
# ── Missing values & duplicates ──────────────────────────────────────────────
missing = df.isnull().sum()
total_missing = missing.sum()
print('='*60)
print('MISSING VALUES PER COLUMN')
print('='*60)
print(missing)
print(f'\nTotal missing values: {total_missing}')

dup_count = df.duplicated().sum()
print(f'\nDuplicate rows: {dup_count}')

MISSING VALUES PER COLUMN
batch_id                      0
instructor_id                 0
course_id                     0
completion_rate               0
avg_score_improvement         0
avg_quiz_score                0
dropout_rate                  0
avg_watch_time                0
assignment_submission_rate    0
forum_activity_rate           0
avg_feedback_score            0
feedback_response_rate        0
dtype: int64

Total missing values: 0

Duplicate rows: 0


In [7]:
# ── Structural facts ─────────────────────────────────────────────────────────
n_instructors = df['instructor_id'].nunique()
n_courses = df['course_id'].nunique()

batches_per_instructor = df.groupby('instructor_id')['batch_id'].count()
min_batches = batches_per_instructor.min()
max_batches = batches_per_instructor.max()
mean_batches = batches_per_instructor.mean()

print(f'Unique instructors: {n_instructors}')
print(f'Unique courses:     {n_courses}')
print(f'Batches per instructor — min: {min_batches}, max: {max_batches}, mean: {mean_batches:.2f}')

Unique instructors: 120
Unique courses:     25
Batches per instructor — min: 7, max: 31, mean: 16.67


## Dataset Structure — Key Facts

- **2,000 rows × 12 columns** — each row is a unique batch–instructor–course record.
- **120 unique instructors** across **25 unique courses**.
- **Batches per instructor**: min = 7, max = 31, mean ≈ 16.67 — the data is reasonably balanced, though some instructors have roughly 4× the observations of others.
- **No missing values** in any column and **no duplicate rows**, so no imputation or de-duplication is needed.
- All numeric features are continuous (float64); the three ID columns (batch, instructor, course) are categorical (object).